## Sentinel-2 L1C

Cloud-native Sentinel-2 workflow: search via STAC, load as lazy xarray, apply Workflow (percentile stretch + gamma + unsharp mask)

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import odc.stac
from box import Box
from pystac_client import Client
from scipy.ndimage import gaussian_filter

from earthatlas import composites

warnings.filterwarnings("ignore", category=UserWarning)

### params

In [ ]:
params = Box(
    {
        "catalog": "https://earth-search.aws.element84.com/v1",
        "collection": "sentinel-2-l2a",
        # "aoi": [-124.0, 47.5, -123.0, 48.0],  # Olympic National Park
        # "toi": "2023-08-13",
        "aoi": [17.4, 46.6, 18.2, 47.1],  # Lake Balaton
        "toi": "2025-12-27",
        "max_cloud": 20,
        "bands": composites.true_color,
        "resolution": 10,
        "chunks": {"x": 2048, "y": 2048},
        "workflow": {
            "lo_pct": 1,
            "hi_pct": 99,
            "gamma": 1.8,
            "alpha": 0.3,
            "sigma": 1.5,
        },
    }
)

### STAC search

In [ ]:
catalog = Client.open(params.catalog)

search = catalog.search(
    collections=[params.collection],
    bbox=params.aoi,
    datetime=params.toi,
    query={"eo:cloud_cover": {"lt": params.max_cloud}},
)

items = list(search.items())
print(f"Found {len(items)} items\n")
for item in items:
    cc = item.properties.get("eo:cloud_cover", "?")
    tile = item.properties.get("s2:mgrs_tile", item.id)
    print(f"{item.datetime:%Y-%m-%d %H:%M}, tile={tile}, cloud={cc:.0f}%")

### Load

In [ ]:
ds = odc.stac.load(
    items,
    bands=params.bands,
    resolution=params.resolution,
    chunks=params.chunks,
    groupby="solar_day",
).isel(time=0)

In [ ]:
ds

In [ ]:
nbytes = sum(ds[b].nbytes for b in params.bands)
print(f"{ds.sizes['y']} x {ds.sizes['x']}")
print(f"{len(params.bands)} bands  {nbytes / 1e9:.2f} GB")

### Workflow

In [ ]:
def percentiles_from_sample(arr, lo_pct, hi_pct):
    """Compute percentiles from strided sample, ignoring nodata (0)."""
    valid = arr[arr > 0]
    p_lo = float(np.percentile(valid, lo_pct))
    p_hi = float(np.percentile(valid, hi_pct))
    return p_lo, p_hi


def workflow(
    ds,
    bands,
    lo_pct=1,
    hi_pct=99,
    gamma=1.8,
    alpha=0.3,
    sigma=1.5,
    strip=1024,
    stride=8,
):
    """stretch -> gamma -> unsharp, returns float32 [0,1] RGB array."""
    #
    sample = ds.isel(x=slice(None, None, stride), y=slice(None, None, stride)).compute()
    percentiles = {}
    for b in bands:
        percentiles[b] = percentiles_from_sample(
            sample[b].values.ravel(), lo_pct, hi_pct
        )
        print(f"  {b}: [{percentiles[b][0]:.0f}, {percentiles[b][1]:.0f}]")

    #
    data = {b: ds[b].values.astype(np.float32) for b in bands}
    H, W = data[bands[0]].shape
    pad = int(sigma * 4)

    result = np.zeros((H, W, len(bands)), dtype=np.float32)
    # NOTE: limit peak memory
    for row in range(0, H, strip):
        r0 = max(0, row - pad)
        r1 = min(H, row + strip + pad)
        top_pad = row - r0
        out_h = min(strip, H - row)

        for i, b in enumerate(bands):
            chunk = data[b][r0:r1].copy()
            p_lo, p_hi = percentiles[b]

            # stretch
            chunk -= p_lo
            chunk /= p_hi - p_lo
            np.clip(chunk, 0, 1, out=chunk)

            # gamma
            np.power(chunk, 1.0 / gamma, out=chunk)

            # unsharp mask
            blur = gaussian_filter(chunk, sigma=sigma)
            chunk += alpha * (chunk - blur)
            np.clip(chunk, 0, 1, out=chunk)
            del blur

            result[row : row + out_h, :, i] = chunk[top_pad : top_pad + out_h]
            del chunk

    return result

In [ ]:
p = params.workflow
rgb = workflow(ds, params.bands, p.lo_pct, p.hi_pct, p.gamma, p.alpha, p.sigma)

### Result

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
ax.imshow(rgb)
ax.set_title(
    f"Workflow: stretch({p.lo_pct},{p.hi_pct})+gamma({p.gamma})+unsharp({p.alpha})"
)
ax.axis("off")
fig.suptitle(
    f"Sentinel-2 - {params.toi[0]}",
    fontsize=14,
    y=1.02,
)
plt.tight_layout()
plt.show()